# IDS 706 — Rust ownership experiments

Adapted from [Kedar-V's course notebook](https://github.com/Kedar-V/data-processing-frameworks-demo/blob/75772a44ed1cbc61e66396fad49a9aa0c913c8ef/notebooks/rust_vs_python_intro.ipynb). The original lesson is retained, with completed **Your turn** cells and short experiment notes.

Select the **Rust** kernel. Three cells intentionally fail to demonstrate immutable assignment, use after a move, and conflicting borrows. Read each error, then continue to the working example. Saved outputs come from running this modified notebook.


# Why is Rust underneath so many fast Python tools?

We just used Polars from Python. One reason it can feel fast is that much of the engine underneath is written in Rust.


Pick the **Rust** kernel (Select Another Kernel, Jupyter Kernel, Rust). If `let` is a `SyntaxError`, you are still on Python. Environment setup is linked in this project's README.

A few cells are supposed to fail. Read the one sentence that matters, then keep going. After the takeaway there is one optional extra about memory.

## First, Rust is still just programming

Before the parts that feel foreign, notice how much of this you already know from Python.


### Hello, Rust

A real Rust program lives in a `.rs` file and starts at `fn main`. Open [the original 01_hello.rs example](https://github.com/Kedar-V/data-processing-frameworks-demo/blob/master/rust/examples/01_hello.rs) if you want to see that. You do not need to compile it.

This notebook is a little different. The kernel will not call `main` for you, so the last line of the next cell does.


In [1]:
// A `.rs` file starts here. `main` is the function the program runs.
fn main() {
    let engine = "Rust"; // `let` names a value
    println!("Hello from {engine}.");
}

main(); // the notebook does not call `main` for you, so we call it here


Hello from Rust.


In a `.rs` file you would stop at the closing brace. `rustc` calls `main` for you.

The kernel compiled that cell, then ran it. A typo would have stopped it before anything printed.

From here on we skip `fn main` and write the lines directly, the same way you typed Python in part 1.


Your turn. Fill in a name, a movie you like, and how many stars you would give it. The `{name}` pieces insert whatever you put in the quotes.


In [2]:
// Sample values for this introductory exercise.
fn main() {
    let name = "Haoran";
    let movie = "Example movie";
    let stars = 4;
    println!("{name}'s review");
    println!("  {movie}");
    println!("  {stars} / 5 stars");
}
main();


Haoran's review


  Example movie


  4 / 5 stars


### Conditions look familiar

Same idea as Python. No colon after the condition, the body goes in `{ }`, and you write `else if` instead of `elif`.

Run this one. We are still talking about a movie rating.


In [3]:
let rating = 4.5;

if rating >= 4.0 {
    println!("{rating} stars: high");
} else if rating >= 3.0 {
    println!("{rating} stars: in the middle");
} else {
    println!("{rating} stars: low");
}


4.5 stars: high


()

Your turn

A movie has `n` ratings. Print `keep` if it has at least 1000, otherwise print `skip`. That is the same cutoff we used in part 1.

Start with `n = 3212` (movie 318). Then try a number under 1000.


In [4]:
// Try a value above the cutoff, then a value below it.
let n = 3212;
if n >= 1000 {
    println!("{n}: keep");
} else {
    println!("{n}: skip");
}
let n = 500;
if n >= 1000 {
    println!("{n}: keep");
} else {
    println!("{n}: skip");
}


3212: keep


500: skip


()

### Loops look familiar too

`0..3` is a range: 0, 1, 2. The 3 is not included. You can also walk a list of ratings, same idea as `for rating in ratings:` in Python.


In [5]:
for n in 0..3 { // 0, 1, 2
    println!("pass {n}");
}

let ratings = [4.5, 3.0, 5.0];
for rating in ratings {
    println!("rating = {rating:.1}");
}


pass 0


pass 1


pass 2


rating = 4.5


rating = 3.0


rating = 5.0


()

Your turn. Count how many of these ratings are 4.0 or higher. `high` should print 3. Put an `if` inside the loop.


In [6]:
let ratings = [4.5, 3.0, 5.0, 2.0, 5.0];
let mut high = 0;
for rating in ratings {
    if rating >= 4.0 {
        high += 1;
    }
}
println!("high ratings: {high}");


high ratings: 3


## Now Rust starts being different

So far Rust mostly looks like Python with braces. This is where the languages start making very different choices.

### Values stay put unless we say otherwise

In Python you can rebind a name whenever you like. `min_ratings = 1000` and later `min_ratings = 2000` is fine.

Rust asks first. A plain `let` holds still. Write `let mut` when the value is supposed to change.

Here the cutoff stays put. The running row count does not.


In [7]:
// No `mut`: this cutoff is not allowed to change.
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

// `mut` means this one is allowed to change. Here, a running count.
let mut rows_read = 0;
for _ in 0..3 { // `_` means we do not use the loop counter
    rows_read += 100_000;
    println!("rows_read = {rows_read}");
}


keep movies with at least 1000 ratings


rows_read = 100000


rows_read = 200000


rows_read = 300000


()

Don't run the next cell yet. In Python this would just work: set the cutoff to 1000, then change it to 2000.

What do you think Rust will do?


In [8]:
let min_ratings = 1000;
println!("keep movies with at least {min_ratings} ratings");

min_ratings = 2000;
println!("keep movies with at least {min_ratings} ratings");


Error: cannot assign twice to immutable variable `min_ratings`

The important line in the error is "cannot assign twice to immutable variable". It names both lines and suggests `mut`.

Your turn. Add `mut` on the `let` line, then uncomment the assignment. If you uncomment first, you get the same error again. That is useful.


In [9]:
// Adding `mut` allows the reassignment that failed in the previous example.
let mut min_ratings = 1000;
println!("before: {min_ratings}");
min_ratings = 2000;
println!("after: {min_ratings}");


before: 1000


after: 2000


**Experiment result:** the immutable example raises E0384. Adding `mut` allows the value to change from 1000 to 2000.

Python never asked which values were supposed to hold still. Nothing in those four lines was "wrong", and that is the problem.

Rust now knows the cutoff was meant to stay put, unless we say otherwise. Next question: who owns a list of ratings?


### Who owns this data?

In Python, `b = a` is a second name for the same list. Both names share it.

In Rust, a list on the heap has one owner. `let moved = ratings` can hand the list over. After that, `ratings` is empty-handed.

You either move it, or you `.clone()` it and pay for a second list.


In [10]:
// `vec!` is a list. These are three movie ratings from part 1.
let ratings = vec![4.5, 3.0, 5.0];
println!("ratings = {ratings:?}"); // `:?` prints the list in a readable way


ratings = [4.5, 3.0, 5.0]


Don't run this yet. After `let moved = ratings`, can you still print both names?


In [11]:
let ratings = vec![4.5, 3.0, 5.0];
let moved = ratings;
println!("{ratings:?} {moved:?}");


Error: borrow of moved value: `ratings`

The error is "borrow of moved value: ratings". The list went to `moved`. Using `ratings` after that is the problem.

The compiler also offers `.clone()`. That is the expensive option: a real second list. The next cell does that so both names can live.


In [12]:
let ratings = vec![4.5, 3.0, 5.0];
let copy = ratings.clone(); // a second list, so both names can live
let moved = ratings; // this move is fine: we still have `copy`
println!("moved = {moved:?}");
println!("copy  = {copy:?}");


moved = [4.5, 3.0, 5.0]


copy  = [4.5, 3.0, 5.0]


Your turn. Delete `.clone()` and run. You should see the same kind of error. Put `.clone()` back so both lines print.

Moving every time would get old. What if we just want to look at the ratings without taking them?


In [13]:
// A clone owns separate data: changing it should not change the original.
let ratings = vec![4.5, 3.0, 5.0];
let mut copy = ratings.clone();
copy.push(1.0);
println!("ratings = {ratings:?}");
println!("copy    = {copy:?}");


ratings = [4.5, 3.0, 5.0]


copy    = [4.5, 3.0, 5.0, 1.0]


**Experiment result:** moving the vector in the earlier example raises E0382 when the original name is used again. Here `.clone()` creates a separate vector: appending 1.0 changes only `copy`, while `ratings` keeps its three original values.

### Borrowing instead of copying

Sometimes we do not want to hand the data over. We just want to let another piece of code use it for a moment.

`&ratings` is a loan. Many readers at once is fine. If somebody is going to write, they need the list to themselves.

Many readers, or one writer. Not both.


In [14]:
let mut ratings = vec![5, 3, 4, 4];
{
    let first = &ratings;
    let second = &ratings;
    println!("two readers: {first:?} and {second:?}");
}
{
    // Take one mutable borrow after the shared borrows are no longer used.
    let writer = &mut ratings;
    writer.push(2);
}
println!("after a write: {ratings:?}");


two readers: [5, 3, 4, 4] and [5, 3, 4, 4]


after a write: [5, 3, 4, 4, 2]


**Experiment result:** two shared references can read the same vector. After their last use, a mutable reference can append 2. The original owner remains `ratings`, and its final contents are `[5, 3, 4, 4, 2]`.

These rules can feel annoying in toy examples. Now let's see the kind of bug they are trying to prevent.

## Why all these rules?

### A bug Python will happily run

Don't run this yet. In Python, removing items from a list while you loop over it often "works". It can also silently skip an item.

Start with `[1, 2, 2, 3, 4, 5]`, drop every 2 while looping, and one 2 survives. No error.

The next cell is the same idea. The `&ratings` is the loan from a minute ago: we are reading the list. Then we try to change it in the same loop.

What do you think Rust will do?

**Modification:** the source code used `for rating in ratings`, which moves the vector. This version uses `for rating in &ratings` and `*rating` so the example tests the shared-versus-mutable borrowing conflict described in the lesson.


In [15]:
// Expected error: modifying a vector while its shared iteration borrow is active.
let mut ratings = vec![1, 2, 2, 3, 4, 5];
for rating in &ratings {
    if *rating == 2 {
        ratings.remove(1);
    }
}
println!("{ratings:?}");


Error: cannot borrow `ratings` as mutable because it is also borrowed as immutable

The important sentence is "cannot borrow `ratings` as mutable because it is also borrowed as immutable". We are reading the list and trying to change it at the same time.

Python runs the same idea and can give the wrong answer, with no error. `ratings.remove(2)` on `[1, 2, 2, 3, 4, 5]` leaves `[1, 2, 3, 4, 5]`. The second 2 was skipped because the list shifted under the loop.

That is what the earlier rules were for. Rust made us state who was reading and who was writing, then refused the program that mixed them.


**Working experiment:** read the original vector through a shared borrow, and write the kept values into a different vector. This avoids changing the vector being read.

In [16]:
let ratings = vec![1, 2, 2, 3, 4, 5];
let mut kept = Vec::new();
for rating in &ratings {
    if *rating != 2 {
        kept.push(*rating);
    }
}
println!("original: {ratings:?}");
println!("filtered: {kept:?}");


original: [1, 2, 2, 3, 4, 5]


filtered: [1, 3, 4, 5]


**Experiment result:** the filtered vector is `[1, 3, 4, 5]`. Both 2s are removed, while the original vector remains `[1, 2, 2, 3, 4, 5]`.

## So what should you remember?

Basic Rust is recognizable if you know Python. `fn main`, `if`, and `for` are the same ideas with different spelling.

Then Rust asks us to be explicit. A plain `let` does not change. A list has one owner. A loan is either shared-and-read or exclusive-and-write.

Python gives more freedom. Some mistakes then show up only when the line runs, or they change the answer and never raise.

Rust rejects some of those programs before they run. That is not a reason to drop Python. Tools like Polars can keep the Python interface and still use this kind of compiled Rust underneath.


### Practical note

You do not need to write Rust to get this. Polars is a Python package. You `import polars`, write Python, and a compiled Rust engine does the heavy work.

As a data scientist you will live in Python most of the time. The useful move is to notice when a library is a thin wrapper around Rust (or C, or C++) and let that engine do the scan, the join, the group-by.

`pip install polars` is that wrapper. The rules we just saw are why the engine underneath can be strict and fast, while the notebook you type in stays Python.


## Optional: where the memory goes

This extra shows a value being freed. `Drop` runs when a value is released.

**Modification:** the allocation was changed from 64 MB to 16 MB. Look for `FREE 16 MB` before the line `back in the outer block`.

In [17]:
// A Buffer is a block of bytes we can watch being freed.
struct Buffer {
    data: Vec<u8>, // the actual bytes
}

fn make_buffer(megabytes: usize) -> Buffer {
    println!("  allocate {megabytes} MB");
    Buffer {
        data: vec![0; megabytes * 1024 * 1024],
    }
}

// `Drop` runs automatically when a Buffer goes out of scope. No `free()` call.
impl Drop for Buffer {
    fn drop(&mut self) {
        let megabytes = self.data.len() / 1024 / 1024;
        println!("  FREE {megabytes} MB");
    }
}

println!("enter outer block");
{
    println!("  enter inner scope");
    let _scratch = make_buffer(16); // create it only to watch it die
    println!("  inner scope is about to end");
} // scratch is freed on this brace
println!("back in the outer block: that memory is already gone");


enter outer block


  enter inner scope


  allocate 16 MB


  inner scope is about to end


  FREE 16 MB


back in the outer block: that memory is already gone


**Experiment result:** the output says `allocate 16 MB` and then `FREE 16 MB` before returning to the outer block. The buffer is dropped when the inner scope ends.

CPython normally uses reference counting and also has a collector for reference cycles. In this Rust example, the buffer has one owner and `Drop` runs when the inner scope ends. The output shows the 16 MB buffer being released before execution returns to the outer block.